In [1]:
# !wget https://ftp.ebi.ac.uk/pub/databases/interpro/current_release/interpro.xml.gz
# !wget ftp://ftp.ebi.ac.uk/pub/databases/interpro/releases/latest/entry.list
# !wget ftp://ftp.ebi.ac.uk/pub/databases/interpro/current_release/protein2ipr.dat.gz

In [1]:
#load data/interpo.xml using xml parsing library
import xml.etree.ElementTree as ET
tree = ET.parse('data/interpro.xml')
root = tree.getroot()

In [2]:
#print number of children of root
interpro_accession_l = []
interpro_member_db_l = []
interpro_type_l = []
prot_count_l = []

count = 0
for entry in root.findall('interpro'):
    memberlist = entry.find('member_list')
    db_entry = [db.get('db') for db in memberlist.findall('db_xref')]
    entry_type = entry.get('type')
    interpro_type_l.append(entry_type)
    interpro_accession_l.append(entry.get('id'))
    interpro_member_db_l.append(db_entry)
    num_prot = int(entry.get('protein_count'))
    prot_count_l.append(num_prot)
    count += 1

from itertools import chain
all_db = set(chain.from_iterable(interpro_member_db_l))
db_cols = {'interpro_accession': interpro_accession_l, 'type': interpro_type_l, 'protein_count': prot_count_l}
for db in all_db:
    db_cols[db] = [True if db in db_entry else False for db_entry in interpro_member_db_l]
import pandas as pd
interpro_df = pd.DataFrame(db_cols)

In [3]:
interpro_df = interpro_df[interpro_df['protein_count'] > 1e4]
# interpro_df = interpro_df[interpro_df['CATHGENE3D']]
# interpro_df

In [4]:
go_annot = pd.read_csv('data/interpro2go', sep=';', header=None, names=['interpro_accession', 'go_id'], comment='!')
go_annot['go_id'] = go_annot['go_id'].str.strip()
go_set = set(go_annot['go_id'])
#using regex, extract id from string of form "InterPro:{ID}..."
go_annot['interpro_accession'] = go_annot['interpro_accession'].str.extract(r'InterPro:(\S+)')
go_annot = go_annot.groupby('interpro_accession').agg({'go_id': list}).reset_index()
interpro_df['go_term'] = interpro_df['interpro_accession'].map(dict(zip(go_annot['interpro_accession'], go_annot['go_id'])))
interpro_df.dropna(subset=['go_term'], inplace=True)
interpro_df = interpro_df[~interpro_df['go_term'].apply(lambda x: len(x) == 1 and 'GO:0005515' in x)]

In [5]:
interpro_df['type'].value_counts()

type
Family                    2301
Domain                    1429
Homologous_superfamily     483
Conserved_site             254
Active_site                 58
Binding_site                41
Repeat                      14
PTM                          2
Name: count, dtype: int64

In [6]:
multidomain_subs = (interpro_df[(interpro_df['type'] == 'Domain') | (interpro_df['type'] == 'Repeat')].sort_values(by='protein_count', ascending=False))
multidomain_subs['term_hash'] = multidomain_subs['go_term'].apply(lambda x: hash(frozenset(x)) % (10 ** 8))

In [7]:
from Bio import SeqIO
swissprot_seq = list(SeqIO.parse('data/uniprot_sprot.fasta', 'fasta'))
swissprot_id = set([rec.id.split('|')[1] for rec in swissprot_seq])

In [8]:
import polars as pl
csv = pl.scan_csv("data/protein2ipr.dat", has_header=False, separator="\t")
q1 = (
    csv.rename({"column_1": "uniprot_id", "column_2": "interpro_accession"})
    .filter(pl.col("interpro_accession").is_in(set(multidomain_subs['interpro_accession'])))
    .filter(pl.col("uniprot_id").is_in(swissprot_id))
    .collect()
)
multidomain_subs = pl.from_dataframe(multidomain_subs)
multidomain_lookup = multidomain_subs[['interpro_accession', 'type', 'protein_count', 'term_hash']]

In [9]:
# Merge q1 with multidomain_lookup on interpro_accession
merged_data = q1.join(
    pl.from_dataframe(multidomain_lookup), 
    on='interpro_accession', 
    how='inner'
)

# Count unique term_hash per protein
protein_term_counts = (merged_data
    .group_by('uniprot_id')
    .agg(pl.col('term_hash').n_unique().alias('unique_term_hash_count'))
)

# Filter out proteins with one or fewer term_hash
proteins_to_keep = protein_term_counts.filter(pl.col('unique_term_hash_count') > 1)

# Filter the merged data to keep only proteins with more than one term_hash
filtered_merged_data = merged_data.join(
    proteins_to_keep.select('uniprot_id'), 
    on='uniprot_id', 
    how='inner'
)

q = filtered_merged_data[['uniprot_id', 'interpro_accession', 'type', 'term_hash']]
q = filtered_merged_data.to_pandas()
seq_len = {rec.id.split('|')[1]: len(rec.seq) for rec in swissprot_seq}
q['seq_len'] = q['uniprot_id'].map(seq_len)
q.rename(columns={'column_5': 'domain_start', 'column_6': 'domain_end'}, inplace=True)

In [ ]:
import numpy as np
from collections import defaultdict
def filter_overlapping_domains(term_hash_l, domain_range_l, seq_len):
    term_hash_dict = defaultdict(np.zeros(seq_len, dtype=bool))
    for term_hash, (si, ei) in zip(term_hash_l, domain_range_l):
        term_hash_dict[term_hash][si:ei] = True
        
    

In [ ]:
next(iter(q.groupby('uniprot_id')))

('A0A017SGC7',
    uniprot_id interpro_accession                        column_3 column_4  \
 0  A0A017SGC7          IPR006094  FAD linked oxidase, N-terminal  PF01565   
 1  A0A017SGC7          IPR012951        Berberine/berberine-like  PF08031   
 2  A0A017SGC7          IPR016166   FAD-binding domain, PCMH-type  PS51387   
 
    domain_start  domain_end    type  protein_count  term_hash  seq_len  
 0            65         198  Domain         269267   49097618      497  
 1           448         489  Domain          45956    3556801      497  
 2            59         229  Domain         341450   75708213      497  )

In [ ]:
import numpy as np
#select proteins annotated with more than two domains 
def domain_overlap(domain_ranges, seq_len):
    domain_count = np.zeros(seq_len, dtype=int)
    domain_mask = np.zeros(seq_len, dtype=bool)
    domain_ranges = sorted(domain_ranges, key=lambda x: x[0])  # Sort by term hash
    phash = None
    for term_hash, domain_range in domain_ranges:
        if (not phash is None) and (phash != term_hash):
            domain_count += domain_mask
            domain_mask.fill(0)
        si, ei = domain_range
        domain_mask[si:ei] = term_hash
        domain_mask[si:ei]



,uniprot_id,interpro_accession,column_3,column_4,domain_start,domain_end,type,protein_count,term_hash,seq_len
0,A0A017SGC7,IPR006094,"FAD linked oxidase, N-terminal",PF01565,65,198,Domain,269267,49097618,497
1,A0A017SGC7,IPR012951,Berberine/berberine-like,PF08031,448,489,Domain,45956,3556801,497
2,A0A017SGC7,IPR016166,"FAD-binding domain, PCMH-type",PS51387,59,229,Domain,341450,75708213,497
3,A0A023I4D6,IPR006094,"FAD linked oxidase, N-terminal",PF01565,67,201,Domain,269267,49097618,512
4,A0A023I4D6,IPR016166,"FAD-binding domain, PCMH-type",PS51387,63,235,Domain,341450,75708213,512
...,...,...,...,...,...,...,...,...,...,...
336347,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,PF00211,874,1059,Domain,161786,33047274,1088
336348,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,PS50125,882,1012,Domain,161786,33047274,1088
336349,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,SM00044,846,1041,Domain,161786,33047274,1088
336350,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,cd07302,881,1058,Domain,161786,33047274,1088


In [ ]:
def domain_overlap(protein_domains):
    protein_domains = protein_domains.sort_values(by='seq_start')
    overlaps = []
    for i in range(len(protein_domains) - 1):
        start1, end1 = protein_domains.iloc[i][['seq_start', 'seq_end']]
        start2, end2 = protein_domains.iloc[i + 1][['seq_start', 'seq_end']]
        if start2 < end1:  # There is an overlap
            overlap_length = min(end1, end2) - start2
            overlaps.append(overlap_length)
    return sum(overlaps)

,uniprot_id,interpro_accession,column_3,column_4,column_5,column_6,type,protein_count,term_hash,seq_len
0,A0A017SGC7,IPR006094,"FAD linked oxidase, N-terminal",PF01565,65,198,Domain,269267,49097618,497
1,A0A017SGC7,IPR012951,Berberine/berberine-like,PF08031,448,489,Domain,45956,3556801,497
2,A0A017SGC7,IPR016166,"FAD-binding domain, PCMH-type",PS51387,59,229,Domain,341450,75708213,497
3,A0A023I4D6,IPR006094,"FAD linked oxidase, N-terminal",PF01565,67,201,Domain,269267,49097618,512
4,A0A023I4D6,IPR016166,"FAD-binding domain, PCMH-type",PS51387,63,235,Domain,341450,75708213,512
...,...,...,...,...,...,...,...,...,...,...
336347,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,PF00211,874,1059,Domain,161786,33047274,1088
336348,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,PS50125,882,1012,Domain,161786,33047274,1088
336349,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,SM00044,846,1041,Domain,161786,33047274,1088
336350,X5M8U1,IPR001054,Adenylyl cyclase class-3/4/guanylyl cyclase,cd07302,881,1058,Domain,161786,33047274,1088


In [35]:
proteins = q.select('uniprot_id').unique().sample(n=250, seed=42)

In [36]:
proteins

uniprot_id
str
"""A0A660ZHL3"""
"""A0A1B1YE92"""
"""A0A6I3ZR94"""
"""A0A0D2X2V2"""
"""A0A2I0DSE7"""
…
"""A0A2N7HV47"""
"""A0A7J7LI46"""
"""A0A1M6A1A2"""


In [7]:
#Build a subset of interpro_df for each of domain, superfamily, family, active_site, and binding_site
#For each subset, randomly select a maximum of 12 entries
superfamily_subs = interpro_df[(interpro_df['type'] == 'Homologous_superfamily' ) & (interpro_df['CATHGENE3D'])].sort_values(by='protein_count', ascending=False).sample(n=12, random_state=1)
domain_subs = interpro_df[interpro_df['type'] == 'Domain'].sort_values(by='protein_count', ascending=False).sample(n=12, random_state=1)
family_subs = interpro_df[interpro_df['type'] == 'Family'].sort_values(by='protein_count', ascending=False).sample(n=12, random_state=1)
active_site_subs = (interpro_df[interpro_df['type'] == 'Active_site'].sort_values(by='protein_count', ascending=False)
                    .sample(n=min(12, (interpro_df['type'] == 'Active_site').sum()), random_state=1))
binding_site_subs = (interpro_df[interpro_df['type'] == 'Binding_site'].sort_values(by='protein_count', ascending=False)
                    .sample(n=min(12, (interpro_df['type'] == 'Binding_site').sum()), random_state=1))
repeat_subs = (interpro_df[interpro_df['type'] == 'Repeat'].sort_values(by='protein_count', ascending=False)
                    .sample(n=min(12, (interpro_df['type'] == 'Repeat').sum()), random_state=1))

In [8]:
subs_df = pd.concat([superfamily_subs, domain_subs, family_subs, active_site_subs, binding_site_subs, repeat_subs])
subs_interpro_ids = set(subs_df['interpro_accession'])
len(subs_interpro_ids)

72

In [ ]:
from Bio import SeqIO
swissprot_seq = list(SeqIO.parse('data/uniprot_sprot.fasta', 'fasta'))
swissprot_id = set([rec.id.split('|')[1] for rec in swissprot_seq])
    .filter(pl.col("swissprot_id").is_in(swissprot_id))


In [11]:
import polars as pl
import gzip
csv = pl.scan_csv("data/protein2ipr.dat", has_header=False, separator="\t")
q1 = (
    csv.rename({"column_1": "swissprot_id", "column_2": "interpro_accession"})
    .filter(pl.col("interpro_accession").is_in(subs_interpro_ids))
    .filter(pl.col("swissprot_id").is_in(swissprot_id))
    .collect()
)
# csv.head(10).collect()

In [12]:
filtered_interpro_map = q1.to_pandas()

In [32]:
filtered_interpro_map['seq'] = filtered_interpro_map['swissprot_id'].map(dict([(rec.id.split('|')[1], str(rec.seq)) for rec in swissprot_seq]))
filtered_interpro_map['seq_len'] = filtered_interpro_map['seq'].apply(len)
filtered_interpro_map.rename(columns={'column_5': 'sind', 'column_6': 'eind'}, inplace=True)
filtered_interpro_map['domain_perc'] = (filtered_interpro_map['eind'] - filtered_interpro_map['sind']) / filtered_interpro_map['seq_len']
filtered_interpro_map = filtered_interpro_map[filtered_interpro_map['seq_len'] < 850]
filtered_interpro_map = filtered_interpro_map[filtered_interpro_map['domain_perc'] < 0.5]
#Group by interpro_accession, sample up to 60 swissprot_id for each interpro_accession, and recombine into a single dataframe
interpro_selection = (filtered_interpro_map.groupby('interpro_accession')
                         .apply(lambda x: x.sample(n=min(40, len(x)), random_state=1))
                         .reset_index(drop=True))
annot_subs_df = pd.merge(interpro_selection, subs_df[['interpro_accession', 'type', 'go_term']], on='interpro_accession', how='left')

/tmp/ipykernel_2791363/3650136495.py:9: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(40, len(x)), random_state=1))


In [33]:
cols = ['UniprotID', 'AnnotatedIndices', 'GOTerm', 'Sequence', 'Type', 'InterproAccession']
annot_subs_df['AnnotatedIndices'] = [[(si, ei)] for si, ei in zip(annot_subs_df['sind'], annot_subs_df['eind'])]
annot_subs_df.rename(columns={'swissprot_id': 'UniprotID', 
                              'go_term': 'GOTerm', 'seq': 'Sequence', 
                              'type': 'Type', 'interpro_accession': 'InterproAccession'}, inplace=True)
annot_subs_df = annot_subs_df[cols]

In [34]:
#group annot_subs_df by UniprotID, InterproAccession, and Type then aggregate AnnotatedIndices and GOTerm into lists, and Sequence into first value
annot_subs_df = (annot_subs_df.groupby(['UniprotID', 'InterproAccession', 'Type'])
                 .agg({'AnnotatedIndices': 'sum', 'GOTerm': 'sum', 'Sequence': 'first'})
                 .reset_index())

In [35]:
for region_type in annot_subs_df['Type'].unique():
    rsub = annot_subs_df[annot_subs_df['Type'] == region_type].copy()
    rsub['p'] = [(a, b) for a, b in zip(rsub['UniprotID'], rsub['InterproAccession'])]
    print(region_type, len(rsub), len(set(rsub['UniprotID'])), len(set(rsub['p'])))

Domain 185 185 185
Repeat 258 252 258
Active_site 465 465 465
Homologous_superfamily 258 258 258
Binding_site 462 462 462
Family 3 3 3


In [36]:
print(annot_subs_df[annot_subs_df['Type'] == 'Family'])

     UniprotID InterproAccession    Type                    AnnotatedIndices  \
341     F4JJE5         IPR001353  Family                         [(30, 101)]   
779     P72942         IPR003400  Family                         [(33, 158)]   
1106    Q555E8         IPR002076  Family  [(47, 171), (52, 173), (172, 240)]   

                                                 GOTerm  \
341                            [GO:0051603, GO:0005839]   
779                            [GO:0022857, GO:0055085]   
1106  [GO:0009922, GO:0016020, GO:0009922, GO:001602...   

                                               Sequence  
341   MSRRYDSRTTIFSPEGRLYQVEYAMEAIGNAGSAIGILAKDGVVLV...  
779   MASSPKAPKSHRKFQSIYHPTRPLSLWQDNQHDQGEVRIEIIPLID...  
1106  METVQSIITEWTDPKSWEKLVQHSFKDSNWKELIDPVNYKFNFGVT...  


In [37]:
for region_type in annot_subs_df['Type'].unique():
    annot_subs_df[annot_subs_df['Type'] == region_type].to_csv(f'datasets/ip_{region_type.lower()}_dataset.csv', index=False, sep='\t')

Family 307 307 307
Homologous_superfamily 493 493 493
Binding_site 299 299 299
Domain 559 559 559
Active_site 646 621 646


In [ ]:
first_entry = root.find('interpro')
print(ET.tostring(first_entry, encoding='utf8').decode('utf8'))